# Dataset build and merge — corrected

**Supersedes the original `dataframe_merger.ipynb`.** Same job: read the twelve
CY2025 Divvy archives, attach weather, produce the analysis table. The logic now
lives in `build_dataset.py` so there is exactly one code path, and this notebook
runs it and checks the result.

Sources, both used as given in the assignment:

* Divvy trips — <https://divvy-tripdata.s3.amazonaws.com/index.html>
* Weather — <https://archive-api.open-meteo.com/v1/archive>

---

## What was wrong, and what replaced it

| # | In the original | Why it fails | Fix |
|---|---|---|---|
| 1 | Weather joined on `started_at` with no timezone handling | **The export is GMT** (`utc_offset_seconds: 0`); `started_at` is local Chicago. Every ride got weather from 5–6 hours earlier. The join matched 100 % of rows and raised nothing. | Pull in UTC, `tz_convert("America/Chicago")`, then join |
| 2 | `pd.read_csv(r"C:\Users\gutti\310Repo\...")` ×13 | Hardcoded to one machine — nobody else can run it, and the rubric wants a peer to reproduce from a clean clone | `build_dataset.py` downloads from source, cached under `data/raw/` |
| 3 | Reads `open-meteo-chicago.csv` | The repo contains `open-meteo_chicago.xlsx` — different name *and* format | Weather fetched from the API; no local file needed |
| 4 | Never aggregates | Output was 5.55 M trip rows with weather bolted on. The brief asks for **rides per local day** | Aggregated to local hour, then local date |
| 5 | `df.drop(columns=["time","started_at","ended_at"])` | Throws away the timestamps the daily target is built from | Timestamps retained through aggregation |
| 6 | `pd.concat([...])` without `ignore_index=True`; no CY2025 filter | Duplicate index values; the 53 rides starting 2024-12-31 stay in the year | `ignore_index=True`, and rides scoped on `started_at` year |

The GMT merge is quantified in section 4 of `02_eda_milestone.ipynb`: mean
absolute error **2.58 °C** in the attached temperature, **12.6 %** of ride-hours
wrong by more than 5 °C, and hourly `corr(temp, rides)` cut from **+0.585** to
**+0.497**.

---
## 1. Run the pipeline

Downloads on first run (~204 MB of Divvy archives) and caches them; later runs
reuse the cache. The weather is read from the committed
`data/weather_hourly_utc.csv` unless that file is deleted.

In [1]:
import subprocess, sys
r = subprocess.run([sys.executable, "build_dataset.py"],
                   capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-2000:])
assert r.returncode == 0, "build_dataset.py failed - see STDERR above"

=== DIVVY TRIPS ===
  months requested: 202501,202502,202503,202504,202505,202506,202507,202508,202509,202510,202511,202512
  raw rows read                5,552,994
  duplicate ride_id                    0
  start month != file month        1,075  (keyed on started_at, so handled correctly)
  rides outside CY2025                53  -> excluded: {datetime.date(2024, 12, 31): 53}
  rides in CY2025              5,552,941
  non-positive duration               29  (flagged, retained)
  null duration                        0
  local hour buckets               8,758
  distinct local dates               365

=== WEATHER (Open-Meteo ERA5 archive) ===
  grid point served: {'latitude': 41.862915, 'longitude': -87.64877, 'elevation': 241.0, 'timezone': 'UTC', 'utc_offset_seconds': 0}
# recorded from the Open-Meteo archive response at pull time, 17 Nov 2026
  rows pulled (UTC, padded)        8,808
  nulls across all variables           0
  DST fall-back doubled hour  ['2025-11-02 01:00:00']
  DST s

---
## 2. Check what it produced

The pipeline asserts internally; these are the same checks stated explicitly so
the notebook shows its work.

In [2]:
import pandas as pd, numpy as np

daily  = pd.read_csv("data/divvy_daily_merged.csv", parse_dates=["date"])
hourly = pd.read_csv("data/divvy_hourly_merged.csv", parse_dates=["hour_local"])

checks = {
    "daily rows == 365":                     len(daily) == 365,
    "hourly rows == 8,759 local hours":      len(hourly) == 8759,
    "CY2025 rides == 5,552,941":             int(daily.rides.sum()) == 5_552_941,
    "hourly total == daily total":           int(hourly.rides.sum()) == int(daily.rides.sum()),
    "no day has zero rides":                 int((daily.rides == 0).sum()) == 0,
    "no weather nulls in merged daily":      int(daily.temperature_2m.isna().sum()) == 0,
    "exactly one 23-hour day (DST)":         int((daily.hours_observed != 24).sum()) == 1,
}
for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
assert all(checks.values()), "a reconciliation check failed"

PASS  daily rows == 365
PASS  hourly rows == 8,759 local hours
PASS  CY2025 rides == 5,552,941
PASS  hourly total == daily total
PASS  no day has zero rides
PASS  no weather nulls in merged daily
PASS  exactly one 23-hour day (DST)


In [3]:
print(open("data/merge_report.txt").read())

=== DIVVY TRIPS ===
  months requested: 202501,202502,202503,202504,202505,202506,202507,202508,202509,202510,202511,202512
  raw rows read                5,552,994
  duplicate ride_id                    0
  start month != file month        1,075  (keyed on started_at, so handled correctly)
  rides outside CY2025                53  -> excluded: {datetime.date(2024, 12, 31): 53}
  rides in CY2025              5,552,941
  non-positive duration               29  (flagged, retained)
  null duration                        0
  local hour buckets               8,758
  distinct local dates               365

=== WEATHER (Open-Meteo ERA5 archive) ===
  grid point served: {'latitude': 41.862915, 'longitude': -87.64877, 'elevation': 241.0, 'timezone': 'UTC', 'utc_offset_seconds': 0}
# recorded from the Open-Meteo archive response at pull time, 17 Nov 2026
  rows pulled (UTC, padded)        8,808
  nulls across all variables           0
  DST fall-back doubled hour  ['2025-11-02 01:00:00']
  DST s

---
## 3. The merged table

`divvy_daily_merged.csv` is the forecasting target: one row per local Chicago
date, `rides` = completed trips whose `started_at` falls on that date. Column
meanings, units and exclusions are in `data/data_description.md`.

In [4]:
display(daily.head(3))
print("columns:", list(daily.columns))
print()
display(daily[["rides","member","casual","temperature_2m","apparent_temperature",
               "precipitation","snowfall","wind_speed_10m"]].describe().T)

,date,rides,member,casual,temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,rain,snowfall,...,cloud_cover,temp_max,temp_min,apparent_max,hours_observed,dow,dow_name,month,is_weekend,precip_any
0,2025-01-01,3562,2321,1241,29.675000,19.570833,68.916667,0.004,0.0,0.028,...,91.750000,33.4,26.5,25.8,24,2,Wednesday,1,False,False
1,2025-01-02,4749,3807,942,27.666667,18.633333,63.416667,0.008,0.0,0.056,...,83.250000,31.5,24.5,22.9,24,3,Thursday,1,False,False
2,2025-01-03,4200,3436,764,22.833333,12.141667,55.375000,0.000,0.0,0.000,...,25.833333,26.0,18.7,16.8,24,4,Friday,1,False,False


columns: ['date', 'rides', 'member', 'casual', 'temperature_2m', 'apparent_temperature', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'snow_depth', 'wind_speed_10m', 'wind_gusts_10m', 'cloud_cover', 'temp_max', 'temp_min', 'apparent_max', 'hours_observed', 'dow', 'dow_name', 'month', 'is_weekend', 'precip_any']



,count,mean,std,min,25%,50%,75%,max
rides,365.0,15213.536986,8551.873212,1024.000000,6683.000000,15360.000000,23321.000000,32308.000000
member,365.0,9735.564384,4741.498114,708.000000,5295.000000,10065.000000,13902.000000,17862.000000
casual,365.0,5477.972603,4375.052730,172.000000,1346.000000,4473.000000,8486.000000,18240.000000
temperature_2m,365.0,51.105202,20.178626,-1.416667,37.333333,52.554167,69.125000,87.570833
apparent_temperature,365.0,47.568671,25.058570,-12.800000,29.641667,47.362500,70.000000,91.491667
precipitation,365.0,0.090003,0.209321,0.000000,0.000000,0.004000,0.063000,1.623000
snowfall,365.0,0.078260,0.367195,0.000000,0.000000,0.000000,0.000000,4.629000
wind_speed_10m,365.0,7.504034,2.842957,2.350000,5.175000,7.083333,9.366667,19.020833


---

Exploratory analysis, the verified-merge evidence, the data-quality summary and
the baseline plan are in **`02_eda_milestone.ipynb`**.